# Stage 2 — Train
**Twitter Sentiment Analysis MLOps · MAI201**

Notebook version of `src/train.py` (DVC stage `train`).

Vectorizes the training text, fits the classifier chosen in `params.yaml`, and
logs **params + train accuracy + model artifact** to MLflow.

**Input:** `data/processed/train.csv` · **Outputs:** `models/model.pkl`, `models/run_id.txt`, one MLflow run

To reproduce the milestone's *baseline + 2 experiments*, run this notebook (plus `03_evaluate.ipynb`) three times, editing `params.yaml` between runs:
1. **Baseline:** `model: logreg`, `ngram_max: 1` (as committed)
2. **Experiment 1:** `featurize.ngram_max: 2`
3. **Experiment 2:** `train.model: nb`

In [ ]:
import os, pickle
import pandas as pd
import yaml
import mlflow
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline

with open("params.yaml", encoding="utf-8") as f:
    params = yaml.safe_load(f)
feat_cfg, train_cfg, ml_cfg = params["featurize"], params["train"], params["mlflow"]
feat_cfg, train_cfg

In [ ]:
mlflow.set_tracking_uri(ml_cfg["tracking_uri"])
mlflow.set_experiment(ml_cfg["experiment_name"])

train_df = pd.read_csv("data/processed/train.csv")
X, y = train_df["text"].astype(str), train_df["label"]
print(f"Training rows: {len(train_df):,}")

## Build vectorizer + model from config

In [ ]:
def build_vectorizer(cfg):
    common = dict(max_features=cfg["max_features"], ngram_range=(1, cfg["ngram_max"]))
    return {"tfidf": TfidfVectorizer, "count": CountVectorizer}[cfg["vectorizer"]](**common)

def build_model(cfg):
    cw = cfg.get("class_weight") or None
    if cfg["model"] == "logreg":
        return LogisticRegression(C=cfg["C"], max_iter=cfg["max_iter"], class_weight=cw)
    if cfg["model"] == "linearsvc":
        return LinearSVC(C=cfg["C"], max_iter=cfg["max_iter"], class_weight=cw)
    if cfg["model"] == "nb":
        return MultinomialNB()
    raise ValueError(cfg["model"])

pipeline = Pipeline([("vectorizer", build_vectorizer(feat_cfg)),
                     ("clf", build_model(train_cfg))])
pipeline

## Fit + log to MLflow

In [ ]:
run_name = f"{train_cfg['model']}-{feat_cfg['vectorizer']}-ng{feat_cfg['ngram_max']}"

with mlflow.start_run(run_name=run_name) as run:
    mlflow.log_params({
        "vectorizer": feat_cfg["vectorizer"],
        "max_features": feat_cfg["max_features"],
        "ngram_max": feat_cfg["ngram_max"],
        "model": train_cfg["model"],
        "C": train_cfg["C"],
        "class_weight": train_cfg.get("class_weight"),
        "train_rows": len(train_df),
    })

    pipeline.fit(X, y)
    train_acc = pipeline.score(X, y)
    mlflow.log_metric("train_accuracy", train_acc)
    mlflow.sklearn.log_model(pipeline, name="model")

    os.makedirs("models", exist_ok=True)
    with open("models/model.pkl", "wb") as f:
        pickle.dump(pipeline, f)
    with open("models/run_id.txt", "w") as f:
        f.write(run.info.run_id)

print(f"run: {run_name}   run_id: {run.info.run_id}   train_accuracy: {train_acc:.4f}")
print("Next: run 03_evaluate.ipynb to attach test metrics to this run.")